# WANDS follow-up: cached FDE diagnostics and GPU baselines

**No catalog re-encoding.** Continue from the completed WANDS MUVERA cost run. First measure DenseOn CPU/GPU search and the missing dense+BM25 → LateOn pipeline, then diagnose FDE quality on 60 seeded queries against the entire catalog.

Five diagnostic controls: existing 4096/8192 CountSketch; the same R8/B16 construction without final compression; paper-style R20/B32 inner projections to 8 or 16 dimensions per bucket (5120/10240 total). No HNSW in these diagnostics. This tests representation quality before more ANN tuning.

Use a GPU runtime. Full-corpus GPU LateOn keeps FP32 token vectors on the GPU; the shared-pool GPU arm uploads only candidate tokens. GPU OOM is reported explicitly. Copying existing caches can take time; subsequent phases reuse them. Measurements run on local disk, with Drive backup between phases.


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

def run_logged(command, *, cwd=None, log_path):
    """Forward child stdout/stderr through notebook output and keep the failure tail."""
    import collections
    import subprocess
    import sys
    from pathlib import Path

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = collections.deque(maxlen=80)
    print(f'Python: {sys.version.split()[0]} | executable: {sys.executable}', flush=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- New invocation ---\n')
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f'Command exited with status {returncode}. Full log: {log_path}\n'
            + ''.join(tail))
    return returncode

drive.mount('/content/drive')
REPO = Path('/content/ras-wands-followup')
BRANCH = 'codex/colbert-muvera-baselines'
LOGS = Path('/content/drive/MyDrive/ras_wands_followup_logs')
if not REPO.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/hanialshater/ras.git', str(REPO)],
               log_path=LOGS / 'setup.log')
else:
    run_logged(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH],
               log_path=LOGS / 'setup.log')
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
run_logged([sys.executable, '-m', 'pip', 'install', '-e',
            str(REPO) + '[dev,wands,wands-systems]'],
           log_path=LOGS / 'install.log')


In [ ]:
import os
os.chdir(REPO)
os.environ.update(PYTHONPATH=str(REPO / 'src') + ':' + str(REPO), USE_TF='0', USE_FLAX='0',
                  PYLATE_SCORES_BACKEND='torch', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
run_logged([sys.executable, '-c', 'import torch, pylate, faiss, psutil; assert torch.cuda.is_available(), "Select GPU runtime"'], cwd=REPO, log_path=LOGS / 'environment.log')
# Includes real CUDA segmented-MaxSim checks when CUDA is available.
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_wands_followup.py'], cwd=REPO, log_path=LOGS / 'tests.log')


## Restore just the required files

Change the two source folders only if your earlier runs used different names. The new output contains small diagnostic score caches and reports, not another copy of the document embeddings. Use a new `RUN_NAME` if code, settings or hardware change.


In [ ]:
import json, shutil
REFERENCE_DRIVE = Path('/content/drive/MyDrive/ras_wands_denseon_lateon_seed7_v1')
CACHE_DRIVE = Path('/content/drive/MyDrive/ras_wands_muvera_cost_seed7_v1')
REFERENCE = Path('/content/wands_systems_reference')
CACHE = Path('/content/ras_wands_muvera_cost_seed7_v1')
RUN_NAME = 'ras_wands_followup_seed7_v1'
RUN = Path('/content') / RUN_NAME
BACKUP = Path('/content/drive/MyDrive') / RUN_NAME
CONFIGS = ['original_4096', 'original_8192', 'raw_r8_b4', 'paper_r20_b5_p8', 'paper_r20_b5_p16']
ARMS = ['dense_cpu', 'dense_gpu', 'late_gpu', 'shared_late_cpu', 'shared_late_gpu']

def copy_files(source, destination, names=None):
    destination.mkdir(parents=True, exist_ok=True)
    names = names if names is not None else [str(p.relative_to(source)) for p in source.rglob('*') if p.is_file()]
    for name in names:
        if '.tmp' in name or name.endswith('.copying'):
            continue
        path, target = source / name, destination / name
        if target.exists() and target.stat().st_size == path.stat().st_size and abs(target.stat().st_mtime-path.stat().st_mtime) < 2:
            continue
        print('Copy:', name, flush=True)
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_suffix(target.suffix + '.copying')
        shutil.copy2(path, temp)
        temp.replace(target)

reference_files = ['dataset.json', 'manifest.json', 'candidate_ids.json', 'quality.csv']
reference_files += [str(p.relative_to(REFERENCE_DRIVE)) for arm in ['dense', 'colbert'] for p in (REFERENCE_DRIVE / arm).glob('scores_*.npy')]
copy_files(REFERENCE_DRIVE, REFERENCE, reference_files)
cache_files = ['manifest.json', 'parity.json', 'serving.json', 'dense/values.npy', 'dense/queries.npy', 'dense/model.json',
               'colbert/values.npy', 'colbert/offsets.npy', 'colbert/queries.npz', 'colbert/model.json']
for dimension in [4096, 8192]:
    if f'original_{dimension}' in CONFIGS:
        cache_files += [f'fde_{dimension}/{name}' for name in ['documents.npy', 'queries.npy', 'map.npz']]
copy_files(CACHE_DRIVE, CACHE, cache_files)
if BACKUP.exists():
    copy_files(BACKUP, RUN)
RUN.mkdir(parents=True, exist_ok=True)
BASE = [sys.executable, '-u', '-m', 'experiments.wands_followup', '--reference-dir', str(REFERENCE),
        '--cache-dir', str(CACHE), '--output-dir', str(RUN), '--device', 'cuda',
        '--configs', *CONFIGS, '--arms', *ARMS, '--diagnostic-queries', '60', '--timing-queries', '30',
        '--timing-repeats', '3', '--threads', '2', '--seed', '7']
def phase(name, *extra):
    try:
        run_logged(BASE + ['--phase', name, *map(str, extra)], cwd=REPO,
                   log_path=LOGS / ('_'.join([name, *map(str, extra)]).replace('--', '') + '.log'))
    finally:
        copy_files(RUN, BACKUP)
phase('prepare')
phase('geometry')


## Serving costs first

Each arm runs in a fresh process. The shared pool is regenerated and checked against the original pool. Full GPU arms keep vectors resident; shared GPU LateOn uploads only candidate tokens. Top-k selection handles document-ID ties without sorting the entire corpus.

Reference score validation occurs **after** timed requests and memory recording so it cannot warm token pages or inflate the reported RAM. Every timed query must pass before a successful latency report is written. CUDA OOM appears as a skipped arm, never a CPU substitute.


In [ ]:
for arm in ARMS:
    phase('serving', '--arm', arm)
phase('report')
import pandas as pd
from IPython.display import display
print((RUN / 'serving_status.json').read_text())
for name in ['latency.csv', 'memory.csv']:
    if (RUN / name).exists():
        display(pd.read_csv(RUN / name))


## FDE diagnostics on the cached embeddings

This is a fixed diagnostic subset against **all products**. Original CountSketch scores are recalculated for those same queries. New maps are applied to the saved token embeddings; only small score shards are retained. This phase can take longer than the serving checks, but it does not run the document models.

Interpretation: if uncompressed R8/B16 recovers quality, final compression is implicated. If it remains poor and R20/B32 inner projections help, partition/repetition settings matter. Neither result alone establishes production cost or generalization. Top-1 candidate recall is reported separately from top-10 overlap to avoid conflating paper metrics.


In [ ]:
for config in CONFIGS:
    phase('diagnose', '--config', config)
phase('report')
for name in ['diagnostic_fidelity.csv', 'diagnostic_quality.csv', 'shared_pool_fidelity_queries.csv']:
    print(name)
    display(pd.read_csv(RUN / name))
print((RUN / 'geometry.json').read_text())


In [ ]:
import zipfile
from google.colab import files
archive = Path('/content') / (RUN_NAME + '_reports.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in RUN.rglob('*'):
        if path.is_file() and path.suffix in ['.csv', '.json']:
            z.write(path, path.relative_to(RUN))
files.download(str(archive))
